In [1]:
!pip install tensorflow


In [ ]:
# install_packages.py
import subprocess
import sys

packages = [
    "tensorflow",    
    "opencv-python",  
    "ultralytics",    
    "matplotlib",     
    "scikit-learn"  
]

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

for pkg in packages:
    print(f"Installing {pkg}...")
    install(pkg)
    print(f"{pkg} installed!\n")

print("All packages installed successfully!")


Installing tensorflow...
tensorflow installed!

Installing opencv-python...
opencv-python installed!

Installing ultralytics...
ultralytics installed!

Installing matplotlib...
matplotlib installed!

Installing scikit-learn...
scikit-learn installed!

All packages installed successfully!


In [ ]:

import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

# YOLO
from ultralytics import YOLO

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Scikit-learn
from sklearn.metrics import confusion_matrix, classification_report

print("All libraries imported successfully!")


All libraries imported successfully!


In [ ]:

# # 2. YOLO Detection & Cropping


def detect_and_crop(images_dir, output_base_dir, model_path, conf_threshold=0.3):
    """
    For each image in `images_dir`, run YOLO detection and crop the detected objects.
    Crops are saved in subfolders of `output_base_dir` by class name.
    """
    # Load YOLO model
    yolo_model = YOLO(model_path)

    # Create output directory if it doesn't exist
    os.makedirs(output_base_dir, exist_ok=True)

    # Process each image
    for img_file in os.listdir(images_dir):
        if not img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
            continue

        img_path = os.path.join(images_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Run YOLO detection
        results = yolo_model.predict(source=img_path, conf=conf_threshold)
        boxes_data = results[0].boxes.data  

        for i, box in enumerate(boxes_data):
            x1, y1, x2, y2, conf, cls_id = box.tolist()
            class_id = int(cls_id)
            # YOLOv8 stores class names in yolo_model.names
            class_name = yolo_model.names.get(class_id, "unknown")

            # Crop the object
            cropped = img[int(y1):int(y2), int(x1):int(x2)]
            if cropped.size == 0:
                continue

            # Create a subfolder for the class
            class_folder = os.path.join(output_base_dir, class_name)
            os.makedirs(class_folder, exist_ok=True)

            # Save the cropped image
            crop_filename = f"{os.path.splitext(img_file)[0]}_{i}.jpg"
            cv2.imwrite(os.path.join(class_folder, crop_filename), cropped)

    print(f"Crops saved to: {output_base_dir}")

print("Function detect_and_crop() defined.")


Function detect_and_crop() defined.


In [ ]:

# # 3. Detect & Crop for Training and Validation Sets


model_path = r"D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\runs\best.pt"

# Training images
train_images_dir = r"D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\dataset\train_augmented\images"
train_output_dir = r"D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\dataset\train_crop"
detect_and_crop(train_images_dir, train_output_dir, model_path, conf_threshold=0.2)

# Validation images
val_images_dir = r"D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\dataset\valid_augmented\images"
val_output_dir = r"D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\dataset\valid_crop"
detect_and_crop(val_images_dir, val_output_dir, model_path, conf_threshold=0.2)



image 1/1 D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\dataset\train_augmented\images\aug_-10_png_jpg.rf.6417436802ce1f127ce36c9037782b98.jpg: 640x640 4 large_debriss, 4 medium_debriss, 384.9ms
Speed: 2.9ms preprocess, 384.9ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\dataset\train_augmented\images\aug_-10_png_jpg.rf.725790c51c80ad4c9308ecf2d650945d.jpg: 640x640 4 large_debriss, 1 medium_debris, 358.6ms
Speed: 2.2ms preprocess, 358.6ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\dataset\train_augmented\images\aug_-10_png_jpg.rf.81030da6da75f06406688a70c28473ca.jpg: 640x640 2 large_debriss, 3 medium_debriss, 306.6ms
Speed: 3.5ms preprocess, 306.6ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

imag

In [ ]:

# # 4. Build & Train ResNet50-based Classifier
# 
# In this cell, we create data generators for our cropped images and build a ResNet50-based

train_dir = train_output_dir
val_dir   = val_output_dir

# Data generator parameters
batch_size = 32
target_size = (128, 128)

# Augmentations for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical'
)
val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical'
)

num_classes = len(train_generator.class_indices)
print("Number of classes:", num_classes)

# Build ResNet50 base
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(target_size[0], target_size[1], 3))
base_model.trainable = False  

# Add custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint(
    'best_resnet_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Train
epochs = 20
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator,
    callbacks=[early_stop, checkpoint]
)


Found 90599 images belonging to 4 classes.
Found 9397 images belonging to 4 classes.
Number of classes: 4
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 134, 134,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 64, 64,    │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 64, 64,    │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 64, 64,    │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 66, 66,    │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 32, 32,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 32, 32,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 32, 32,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 32, 32,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 32, 32,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 32, 32,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 32, 32,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 32, 32,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 32, 32,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 32, 32,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 32, 32,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,500 (90.98 MB)

 Trainable params: 262,788 (1.00 MB)

...

2832/2832 ━━━━━━━━━━━━━━━━━━━━ 800s 282ms/step - accuracy: 0.9129 - loss: 0.2622 - val_accuracy: 0.9397 - val_loss: 0.2176
Epoch 9/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.9123 - loss: 0.2630
Epoch 9: val_accuracy did not improve from 0.93966
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 796s 281ms/step - accuracy: 0.9123 - loss: 0.2630 - val_accuracy: 0.9394 - val_loss: 0.2038
Epoch 10/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step - accuracy: 0.9131 - loss: 0.2638
Epoch 10: val_accuracy improved from 0.93966 to 0.94019, saving model to best_resnet_model.h5


2832/2832 ━━━━━━━━━━━━━━━━━━━━ 1098s 388ms/step - accuracy: 0.9131 - loss: 0.2638 - val_accuracy: 0.9402 - val_loss: 0.2073
Epoch 11/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - accuracy: 0.9116 - loss: 0.2665
Epoch 11: val_accuracy improved from 0.94019 to 0.94062, saving model to best_resnet_model.h5


2832/2832 ━━━━━━━━━━━━━━━━━━━━ 874s 309ms/step - accuracy: 0.9116 - loss: 0.2665 - val_accuracy: 0.9406 - val_loss: 0.2036
Epoch 12/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step - accuracy: 0.9120 - loss: 0.2651
Epoch 12: val_accuracy improved from 0.94062 to 0.94115, saving model to best_resnet_model.h5


2832/2832 ━━━━━━━━━━━━━━━━━━━━ 903s 319ms/step - accuracy: 0.9120 - loss: 0.2651 - val_accuracy: 0.9412 - val_loss: 0.2117
Epoch 13/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - accuracy: 0.9148 - loss: 0.2564
Epoch 13: val_accuracy did not improve from 0.94115
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 926s 327ms/step - accuracy: 0.9148 - loss: 0.2564 - val_accuracy: 0.9406 - val_loss: 0.2074
Epoch 14/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - accuracy: 0.9129 - loss: 0.2612
Epoch 14: val_accuracy did not improve from 0.94115
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 1393s 492ms/step - accuracy: 0.9129 - loss: 0.2612 - val_accuracy: 0.9406 - val_loss: 0.1995
Epoch 15/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step - accuracy: 0.9146 - loss: 0.2614
Epoch 15: val_accuracy did not improve from 0.94115
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 831s 293ms/step - accuracy: 0.9146 - loss: 0.2614 - val_accuracy: 0.9390 - val_loss: 0.2012
Epoch 16/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - accuracy: 0.9128 - 

2832/2832 ━━━━━━━━━━━━━━━━━━━━ 893s 315ms/step - accuracy: 0.9117 - loss: 0.2608 - val_accuracy: 0.9414 - val_loss: 0.2005
Epoch 19/20
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - accuracy: 0.9120 - loss: 0.2603
Epoch 19: val_accuracy did not improve from 0.94136
2832/2832 ━━━━━━━━━━━━━━━━━━━━ 927s 327ms/step - accuracy: 0.9120 - loss: 0.2603 - val_accuracy: 0.9380 - val_loss: 0.2014


In [ ]:

# # 5. Evaluate & Save Final Model

val_generator.reset()
Y_pred = model.predict(val_generator, steps=len(val_generator))
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_generator.classes

print("Unique classes in y_true:", np.unique(y_true))

# Build target_names from the training generator
target_names = [None] * num_classes
for class_name, index in train_generator.class_indices.items():
    target_names[index] = class_name

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
cr = classification_report(y_true, y_pred, labels=list(range(num_classes)), target_names=target_names, zero_division=0)
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", cr)

# Save the final model
save_dir = r"D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\models\cnn"
os.makedirs(save_dir, exist_ok=True)
model_save_path = os.path.join(save_dir, "debris_resnet_classifier.h5")
model.save(model_save_path)
print(f"Final ResNet-based model saved to {model_save_path}")


294/294 ━━━━━━━━━━━━━━━━━━━━ 57s 191ms/step


Unique classes in y_true: [0 1 2 3]
Confusion Matrix:
 [[7777    0    0  628]
 [  53    0    0    3]
 [   5    0    0    1]
 [ 859    0    0   71]]
Classification Report:
                precision    recall  f1-score   support

 large_debris       0.89      0.93      0.91      8405
medium_debris       0.00      0.00      0.00        56
       rocket       0.00      0.00      0.00         6
    satellite       0.10      0.08      0.09       930

     accuracy                           0.84      9397
    macro avg       0.25      0.25      0.25      9397
 weighted avg       0.81      0.84      0.82      9397

Final ResNet-based model saved to D:\Canada\Subjects\Semester -1\AIDI 1003_01_CAPSTONE TERM 1\Cosmic_Navigators_Final\models\cnn\debris_resnet_classifier.h5
